This code uses a large language model (LLM) to automatically assign top-3 category tags to support tickets. First, zero-shot classification is applied, where the model predicts tags without seeing any examples. Then, few-shot learning is used by adding sample tickets and their labels in the prompt, which helps the model better understand context and improves tagging accuracy. Both approaches output the three most probable tags with confidence scores, allowing comparison between zero-shot and few-shot performance without fine-tuning the model.

In [1]:
import pandas as pd
from transformers import pipeline


In [2]:
data = {
    "ticket_text": [
        "Internet connection is very slow and keeps disconnecting",
        "I was charged twice for my monthly bill",
        "Unable to login to my account after password reset",
        "My router is not working after the update",
        "Requesting refund for cancelled subscription"
    ]
}

df = pd.DataFrame(data)
df


,ticket_text
0,Internet connection is very slow and keeps dis...
1,I was charged twice for my monthly bill
2,Unable to login to my account after password r...
3,My router is not working after the update
4,Requesting refund for cancelled subscription


In [3]:
labels = [
    "Billing Issue",
    "Technical Issue",
    "Account Access",
    "Network Problem",
    "Refund Request",
    "Service Cancellation"
]


In [4]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)


config.json: 0.00B [00:00, ?B/s]

c:\Users\tahir\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tahir\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [5]:
def zero_shot_tagging(text):
    result = classifier(text, labels, multi_label=True)
    
    top_3 = list(zip(
        result["labels"][:3],
        result["scores"][:3]
    ))
    
    return top_3


In [6]:
df["zero_shot_tags"] = df["ticket_text"].apply(zero_shot_tagging)
df


,ticket_text,zero_shot_tags
0,Internet connection is very slow and keeps dis...,"[(Network Problem, 0.9377893209457397), (Techn..."
1,I was charged twice for my monthly bill,"[(Billing Issue, 0.9788798689842224), (Refund ..."
2,Unable to login to my account after password r...,"[(Account Access, 0.9452186822891235), (Techni..."
3,My router is not working after the update,"[(Network Problem, 0.9769672751426697), (Techn..."
4,Requesting refund for cancelled subscription,"[(Refund Request, 0.9858006834983826), (Servic..."


In [7]:
few_shot_examples = """
Ticket: Internet is not working properly
Tags: Network Problem, Technical Issue

Ticket: I cannot access my account
Tags: Account Access

Ticket: I was billed incorrectly
Tags: Billing Issue
"""


In [8]:
def few_shot_prompt(ticket):
    prompt = f"""
You are a support ticket classification system.

Categories:
{labels}

Examples:
{few_shot_examples}

Now classify the following ticket.
Return the top 3 most relevant categories.

Ticket: {ticket}
"""
    return prompt


In [9]:
def few_shot_tagging(ticket):
    result = classifier(
        few_shot_prompt(ticket),
        labels,
        multi_label=True
    )

    top_3 = list(zip(
        result["labels"][:3],
        result["scores"][:3]
    ))
    
    return top_3


In [10]:
df["few_shot_tags"] = df["ticket_text"].apply(few_shot_tagging)
df


,ticket_text,zero_shot_tags,few_shot_tags
0,Internet connection is very slow and keeps dis...,"[(Network Problem, 0.9377893209457397), (Techn...","[(Network Problem, 0.6919825077056885), (Techn..."
1,I was charged twice for my monthly bill,"[(Billing Issue, 0.9788798689842224), (Refund ...","[(Billing Issue, 0.8540059924125671), (Service..."
2,Unable to login to my account after password r...,"[(Account Access, 0.9452186822891235), (Techni...","[(Account Access, 0.9024816751480103), (Techni..."
3,My router is not working after the update,"[(Network Problem, 0.9769672751426697), (Techn...","[(Network Problem, 0.6956244707107544), (Techn..."
4,Requesting refund for cancelled subscription,"[(Refund Request, 0.9858006834983826), (Servic...","[(Service Cancellation, 0.9330256581306458), (..."
